# Module 9 · 文字 3：上下文嵌入與句向量（BERT / Sentence-Transformers）

> **本節定位｜2026 主流核心**
> `02` 把文字變成 `input_ids`，本節把 `input_ids` 餵進 Transformer，
> 得到**隨上下文變化**的嵌入向量。這是現代語意理解、語意檢索、RAG 的共同基礎。

## 學習目標
- 理解 **上下文嵌入 (contextual embeddings)** 與靜態詞向量的根本差異（一詞多義）。
- 用 `AutoModel` 取得 token 級嵌入 **`last_hidden_state (B, L, D)`**，並用 **CLS / mean pooling** 得到句向量 `(B, D)`。
- 用 `sentence-transformers` 一行得到高品質**句向量**，並做**語意檢索 (semantic search)**——RAG 的核心積木。
- 看清楚「資料結構」：token 嵌入 `(B, L, D)` → 句向量 `(B, D)`。

## 0. 資料結構設計：嵌入的形狀

| 物件 | 形狀 | 說明 |
|:--|:--|:--|
| `input_ids` / `attention_mask` | `(B, L)` | 上一節 tokenizer 的輸出 |
| `last_hidden_state`（token 嵌入） | `(B, L, D)` | 每個 token 一個 D 維向量（D≈384/768） |
| 句向量（pooled） | `(B, D)` | 把 L 個 token 嵌入池化成「一句一向量」 |
| 相似度檢索結果 | `(N_query, top_k)` | 每個 query 取回最相近的 k 篇 |

池化（pooling）方式：**CLS pooling**（取 `[CLS]` 位置）或 **mean pooling**（對有效 token 平均，需用 `attention_mask` 排除 padding）。句向量任務通常 mean pooling 較穩。

In [ ]:
try:
    import torch
    from transformers import AutoTokenizer, AutoModel
    HAS_HF = True
except Exception:
    HAS_HF = False
    print("未安裝 transformers/torch，請 `uv sync --extra multimodal`。說明仍可閱讀。")

## 1. 同一個詞，不同上下文 → 不同向量

經典靜態詞向量裡 "bank" 只有一個向量；上下文嵌入會根據整句給出**不同**的 "bank" 向量。

In [ ]:
if HAS_HF:
    tok = AutoTokenizer.from_pretrained("bert-base-uncased")
    model = AutoModel.from_pretrained("bert-base-uncased")
    model.eval()

    s1 = "I deposited money in the bank."   # 銀行
    s2 = "We sat on the river bank."         # 河岸

    def bank_vector(sentence):
        enc = tok(sentence, return_tensors="pt")
        with torch.no_grad():
            out = model(**enc)
        hidden = out.last_hidden_state[0]               # (L, D)
        tokens = tok.convert_ids_to_tokens(enc["input_ids"][0])
        idx = tokens.index("bank")
        return hidden[idx]                              # (D,)

    v1, v2 = bank_vector(s1), bank_vector(s2)
    cos = torch.nn.functional.cosine_similarity(v1, v2, dim=0)
    print(f"last_hidden_state 形狀範例: (L, D)，D = {v1.shape[0]}")
    print(f"兩個 'bank' 的餘弦相似度 = {cos.item():.3f}")
    print("→ 明顯 < 1：同一個詞因上下文不同而得到不同向量（靜態詞向量做不到）。")

## 2. 從 token 嵌入到句向量：mean pooling

把 `(B, L, D)` 的 token 嵌入，用 `attention_mask` 排除 padding 後平均，得到句向量 `(B, D)`。

In [ ]:
if HAS_HF:
    def mean_pool(last_hidden, mask):
        mask = mask.unsqueeze(-1).float()               # (B, L, 1)
        summed = (last_hidden * mask).sum(dim=1)        # (B, D)
        counts = mask.sum(dim=1).clamp(min=1e-9)        # (B, 1)
        return summed / counts

    sentences = ["The movie was fantastic!", "What a terrible film.", "An amazing masterpiece."]
    enc = tok(sentences, padding=True, return_tensors="pt")
    with torch.no_grad():
        out = model(**enc)
    sent_emb = mean_pool(out.last_hidden_state, enc["attention_mask"])
    print("token 嵌入 last_hidden_state :", tuple(out.last_hidden_state.shape), "(B, L, D)")
    print("池化後句向量 sentence_emb     :", tuple(sent_emb.shape), "(B, D)")

## 3. Sentence-Transformers：一行得到高品質句向量

直接用 BERT mean-pooling 的句向量品質有限。`sentence-transformers` 提供**專為句向量微調過**的模型
（如 `all-MiniLM-L6-v2`，384 維、體積小、CPU 友善），是業界做語意檢索/RAG 的常用選擇。

In [ ]:
if HAS_HF:
    try:
        from sentence_transformers import SentenceTransformer
        st = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
        emb = st.encode(sentences)                      # (N, 384) numpy
        print("句向量形狀 :", emb.shape, "(N, D=384)")
    except Exception as e:
        print("（未安裝 sentence-transformers，略過）", e)

## 4. 語意檢索 (Semantic Search)：RAG 的核心積木

把語料庫每篇都編成向量，query 也編成向量，用**餘弦相似度**取回最相近的幾篇。
這就是 RAG（檢索增強生成）「檢索」階段的本質。

In [ ]:
if HAS_HF:
    try:
        from sentence_transformers import SentenceTransformer, util
        st = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
        corpus = [
            "Python is a popular programming language.",
            "The Eiffel Tower is located in Paris.",
            "Transformers revolutionized natural language processing.",
            "Cats are independent and curious animals.",
            "BERT and GPT are based on the Transformer architecture.",
        ]
        corpus_emb = st.encode(corpus, convert_to_tensor=True)   # (N, D)
        query = "What model architecture do modern language models use?"
        q_emb = st.encode(query, convert_to_tensor=True)
        scores = util.cos_sim(q_emb, corpus_emb)[0]              # (N,)
        topk = scores.topk(2)
        print(f"Query: {query}\n")
        for score, idx in zip(topk.values, topk.indices):
            print(f"  {score.item():.3f} | {corpus[idx]}")
        print("\n→ 即使 query 完全沒出現 'Transformer'，仍能憑語意找回相關句子。")
    except Exception as e:
        print("（未安裝 sentence-transformers，略過）", e)

## 小結

| 重點 | 內容 |
|:--|:--|
| 上下文嵌入 | 同詞依上下文得到不同向量，解決一詞多義 |
| token 嵌入 | `last_hidden_state (B, L, D)` |
| 句向量 | CLS / mean pooling → `(B, D)`；句向量任務優先用 `sentence-transformers` |
| 語意檢索 | 向量化 + 餘弦相似度 top-k，是 **RAG 檢索**的本質 |

**下一節 `04_llm_data_formats`**：要訓練/微調**生成式 LLM**，資料不再只是
「一句一向量」，而是 **chat / instruction 的 JSONL**，還要做去重與品質過濾。